In [ ]:
# Install packages
from pathlib import Path
import mne
import numpy as np
from scipy.io import loadmat

# Global variables setup
PATH = Path("/Users/emmaabecasis/Downloads/meditation_test_data") # make sure this fits
# may have to edit, depending on file format

In [ ]:
# Import files
if PATH.is_dir():
    items = list(PATH.rglob("*"))

    edfs = [p for p in items if p.is_file() and p.suffix.lower() == ".edf"]
    print("EDF files found:", len(edfs))
    data_files = [str(file_path.resolve()) for file_path in edfs]

In [ ]:
# If using trigger hub/adding triggers in DSI-Streamer
event_id = {
  "pre_meditation": 1,
  "meditation": 2,
  "post_meditation": 3,
}

# We can also manually add annotations if using standardized procedure
annot = mne.Annotations(
    onset=[0, 60, 120], # Edit to match actual times in seconds
    duration=[60, 60, 60], # Edit to match actual durations in seconds
    description=["pre_meditation", "meditation", "post_meditation"],
  )

In [ ]:
def detect_eyeblink(raw, interactive_annot):
    eog_events = mne.preprocessing.find_eog_events(raw)
    onsets = eog_events[:, 0] / raw.info["sfreq"] - 0.25
    durations = [0.5] * len(eog_events)
    descriptions = ["bad blink"] * len(eog_events)
    blink_annot = mne.Annotations(
        onsets, durations, descriptions, orig_time=raw.info["meas_date"]
    )
    raw.set_annotations(blink_annot + interactive_annot)

def load_edf(file_path):
    data = mne.io.read_raw_edf(file_path, preload=True)

    # Reference to left ear channel
    data.set_eeg_reference(ref_channels="EEG LE-Pz")

    # Standardize channel names
    rename_dict = {}
    for ch_name in data.ch_names:
        if '-Pz' in ch_name:
            rename_dict[ch_name] = ch_name.replace('-Pz', '')
        if ch_name == "Pz":
            rename_dict[ch_name] = "EEG Pz"
    data.rename_channels(rename_dict)

    # create MNE raw object (should be channels x samples - otherwise need to transpose data)
    raw = mne.io.RawArray(data.get_data().T, data.info)
    raw.set_annotations(annot)
    interactive_annot = raw.annotations

    # visually inspect data - annotate bad time spans and channels
    # press A to enter annotation mode - close window to save annotations
    fig = raw.plot(show_annotations=True, scalings="auto", n_channels=7, duration=60)

    # automatically detect eye blinks
    detect_eyeblink(raw, interactive_annot)

    # band pass filter
    raw.filter(l_freq=0.5, h_freq=41)

    # downsample to 256 HA
    raw.resample(sfreq=256)

    return raw

# Artifact removal
from mne.preprocessing import ICA

def ica(raw):
    filt_raw = raw.copy().filter(l_freq=1.0, h_freq=None)
    ica = ICA(max_iter="auto", range=97)
    ica.fit(filt_raw)
    ica.plot_sources(raw, show_scrollbars=False)
    ica.plot_components()
    return ica

# arbitrary epoch rejection criteria used; we can change if this doesn't fit our data
reject_criteria = dict(
    mag=3000e-15,  # 3000 fT
    grad=3000e-13,  # 3000 fT/cm
    eeg=100e-6,  # 100 µV
    eog=200e-6,   # 200 µV
)  

flat_criteria = dict(mag=1e-15, grad=1e-13, eeg=1e-6)  # 1 fT  # 1 fT/cm  # 1 µV

def segment_epochs(raw, epoch_duration=5):
    events = mne.find_events(raw, stim_channel="Trigger")

    # Keep only the pre-meditation, meditation, and post-meditation markers
    stage_codes = set(event_id.values())
    events = events[np.isin(events[:, 2], list(stage_codes))]
    events = events[np.argsort(events[:, 0])]

    sfreq = raw.info["sfreq"]
    samples_per_epoch = round(epoch_duration * sfreq)
    epoch_events = []

    for i, (stage_start, _, code) in enumerate(events):
        stage_end = (
            events[i + 1, 0]
            if i + 1 < len(events)
            else raw.first_samp + raw.n_times
        )

        # Add an epoch only if the entire epoch fits within this stage
        starts = np.arange(
            stage_start,
            stage_end - samples_per_epoch + 1,
            samples_per_epoch,
        )
        epoch_events.extend(
            [[sample, 0, code] for sample in starts]
        )

    if not epoch_events:
        raise ValueError("No complete 5-second epochs fit within the marked stages.")

    return mne.Epochs(
        raw,
        np.asarray(epoch_events, dtype=int),
        event_id=event_id,
        tmin=0,
        tmax=(samples_per_epoch - 1) / sfreq,
        reject = reject_criteria,
        flat = flat_criteria,
        baseline=None,
        preload=True,
    )

def quality_check(epochs, participant_id):
    original_epoch_count = len(epochs.events)
    epochs.drop_bad()
    retained_epoch_count = len(epochs)
    rejected_epoch_count = original_epoch_count - retained_epoch_count
    
    if original_epoch_count:
        rejected_percentage = (100* rejected_epoch_count / original_epoch_count)
    else:
        rejected_percentage = np.nan
    
    print(f"\n{participant_id} quality summary")
    print(f"Original epochs: {original_epoch_count}")
    print(f"Retained epochs: {retained_epoch_count}")
    print(f"Rejected epochs: {rejected_epoch_count}")
    print(f"Rejected percentage: {rejected_percentage:.1f}%")
    
    epochs.plot_drop_log()
    epochs.plot(n_epochs=min(10, len(epochs)), scalings="auto")
    epochs.compute_psd().plot()
 
    return epochs


In [ ]:
from mne.preprocessing import corrmap

icas = []
raws = []
epochs_list = []

for i, file in enumerate(data_files):
    raw = load_edf(file)
    raws.append(raw)
    ica = ica(raw)
    icas.append(ica)

    if i == 0:
        ica.plot_sources(raw, show_scrollbars=False)

        # manually select which components to exclude
        ica.exclude = [0, 1]  # example: exclude the first two components

    else:
        raw = raws[0]
        ica = icas[0]
        eog_inds, eog_scores = ica.find_bads_eog(raw, ch_name="Fpz")
        corrmap(icas, template=(0, eog_inds[0]))

    raw.fit(ica)
    raws[i] = raw

    epochs = segment_epochs(raw)
    quality_check(epochs, file)

    epochs_list.append(epochs)